# Preprocessing

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# 1. Đọc dữ liệu vô
df = pd.read_csv('../data/raw/ecg.csv', header=None)

# 2. Tách đặc trưng (X) và nhãn (y)
# Cột cuối cùng (140) là nhãn, từ 0 đến 139 là tín hiệu
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# 3. Chia dữ liệu: Train (70%), Val (15%), Test (15%)
# Lần 1: Cắt 70% cho Train, 30% còn lại cho Val & Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Lần 2: Chẻ đôi cái 30% kia ra làm Val và Test (mỗi thằng 15%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# 4. Chuẩn hóa dữ liệu (Scale về khoảng [0, 1])
# Fit trên tập Train thôi, rồi apply cho Val với Test để tránh rò rỉ dữ liệu (data leakage)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Kích thước tập Train: {X_train_scaled.shape}")
print(f"Kích thước tập Val:   {X_val_scaled.shape}")
print(f"Kích thước tập Test:  {X_test_scaled.shape}")

Kích thước tập Train: (3498, 140)
Kích thước tập Val:   (750, 140)
Kích thước tập Test:  (750, 140)


In [4]:
# Lọc ra chỉ lấy dữ liệu bình thường (nhãn 1.0) để train con Autoencoder
train_normal_indices = (y_train == 1.0)
X_train_autoencoder = X_train_scaled[train_normal_indices]

print(f"Kích thước Train (chỉ lấy Bình thường): {X_train_autoencoder.shape}")

Kích thước Train (chỉ lấy Bình thường): (2043, 140)


# Lưu tập tin

In [5]:
# Nhập các thư viện cần thiết cho quá trình xử lý và lưu trữ
import pandas as pd
import numpy as np
import os

# Tạo thư mục chứa dữ liệu đã qua xử lý nếu hệ thống chưa có
os.makedirs('../data/preprocessed', exist_ok=True)

# 1. Lưu tập huấn luyện (Train)
# Lưu ý: Tập này chỉ chứa dữ liệu bình thường (nhãn 1.0) đã được chuẩn hóa.
# Khôi phục nhãn 1.0 để đồng nhất cấu trúc dữ liệu khi huấn luyện mô hình.
df_train_processed = pd.DataFrame(X_train_autoencoder)
df_train_processed['label'] = 1.0
df_train_processed.to_csv('../data/preprocessed/train_normal.csv', index=False, header=False)

# 2. Lưu tập xác thực (Validation)
# Tập này chứa dữ liệu đã chuẩn hóa và ghép lại với nhãn tương ứng ban đầu.
df_val_processed = pd.DataFrame(X_val_scaled)
df_val_processed['label'] = y_val
df_val_processed.to_csv('../data/preprocessed/val.csv', index=False, header=False)

# 3. Lưu tập kiểm tra (Test)
# Tập này chứa dữ liệu đã chuẩn hóa và ghép lại với nhãn tương ứng ban đầu.
df_test_processed = pd.DataFrame(X_test_scaled)
df_test_processed['label'] = y_test
df_test_processed.to_csv('../data/preprocessed/test.csv', index=False, header=False)

print("Đã hoàn tất lưu các tập dữ liệu: train_normal.csv, val.csv, và test.csv vào thư mục '../data/preprocessed/'.")

Đã hoàn tất lưu các tập dữ liệu: train_normal.csv, val.csv, và test.csv vào thư mục '../data/preprocessed/'.
